Note: 

Both SwapRateMCSimulator and SwapRatePCAEstimator require historical swap rate time series data in a specific format.
If no explicit file path is provided to the constructor, a default CSV file will be used instead.

The default dataset is located in the "curve_data" folder of the project (see "eur_swap_ts.csv" for the expected format).

In the examples below, a file path is explicitly provided, although it points to the same data as the default.

This notebook demonstrates how to:
- create a liability portfolio consisting of liability and euribor6m irs positions
- simulate PnL for the portfolio, based mc scenarios for the discounting curve
- calculate risk sensitivities, mc CVaR and related metrics

In [1]:
from __future__ import annotations
import os 
from pathlib import Path

# It is assumed that the notebook is executed from within the project - either root or sub folder. 
CWD_PATH = Path.cwd()
SWAP_TIME_SERIES_CSV_PATH = ""

for parent in CWD_PATH.parents:
 if (parent / "src").exists():
     SWAP_TIME_SERIES_CSV_PATH = parent / "curve_data" / "eur_swap_ts.csv"
     break       
 else:
     SWAP_TIME_SERIES_CSV_PATH = CWD_PATH / "curve_data" / "eur_swap_ts.csv"

print(CWD_PATH)
print(SWAP_TIME_SERIES_CSV_PATH)   

C:\Users\mduba\Desktop\Programming\Python\alm_ir_hedging_prototype\example_notebooks
C:\Users\mduba\Desktop\Programming\Python\alm_ir_hedging_prototype\curve_data\eur_swap_ts.csv


In [21]:
# imports 
from typing import Dict
from datetime import datetime 
import numpy as np
import pandas as pd 
from rateslib import IRS as rlIRS, dcf as rlDCF 
from curve_engine.swap_curve_solver import SwapCurveSolver
from curve_engine.swap_curve_builder import SwapCurveBuilder
from curve_engine.swap_rate_pca_estimator import SwapRatePCAEstimator
from curve_engine.swap_rate_mc_simulator import SwapRateMCSimulator
from curve_engine.swap_curve_scenario_generator import SwapCurveScenarioGenerator
from position_engine.euribor6m_irs_position import Euribor6mIRSPosition
from position_engine.liability_position import LiabilityPosition 
from position_engine.liability_portfolio import LiabilityPortfolio

In [4]:
# parameters 
curve_date = datetime(2026,3,31)

swap_rates =  {
 '2Y' : 2.948 , 
 '3Y' : 2.934, 
 '5Y' : 2.983, 
 '7Y' : 3.036, 
 '10Y': 3.11,  
 '12Y': 3.195,  
 '15Y': 3.2439, 
 '20Y': 3.246, 
 '25Y': 3.186,
 '30Y': 3.115 }
 
liabilities_payment_schedule = {
 '1Y': 240716648,   '2Y': 249313671,  '3Y': 260776369,  '4Y': 275104740,   
 '5Y': 292298787,   '6Y': 312358507,  '7Y': 335283902,  '8Y': 361074972, 
 '9Y': 384000367,  '10Y': 406925762, '11Y': 395463064, '12Y': 418388459,
 '13Y': 444179529, '14Y': 472836273, '15Y': 504358691, '16Y': 535881109, 
 '17Y': 561672178, '18Y': 578866225, '19Y': 587463248, '20Y': 590328922,
 '21Y': 584597573, '22Y': 590328922, '23Y': 584597573, '24Y': 573134876,
 '25Y': 555940830, '26Y': 533015435, '27Y': 504358691, '28Y': 469970598, 
 '29Y': 429851157, '30Y': 389731716 } 

In [5]:
# create a solver - enables calibration of curve to swap rates
solver = SwapCurveSolver(
 solver_id = 'eurswap_solver', 
 curve_date = curve_date, 
 instrument_rates = swap_rates
)

SUCCESS: `func_tol` reached after 7 iterations (levenberg_marquardt), `f_val`: 4.554749889445546e-16, `time`: 0.0984s


In [6]:
# construct the curve       
curve = SwapCurveBuilder.from_solver(
 curve_id = 'eurswap', 
 curve_solver = solver
)

In [7]:
# create the par -> forward Jacobian for the curve - needed to derive forward swap rates and calculate forward PV01
curve.calc_par_fwd_jacobian()

SUCCESS: `func_tol` reached after 0 iterations (levenberg_marquardt), `f_val`: 0.0, `time`: 0.0063s


C:\Users\mduba\Desktop\Programming\Python\alm_ir_hedging_prototype\.venv\Lib\site-packages\rateslib\curves\curves.py:275: UserWarning: Evaluating points on a curve beyond the endpoint of the basic spline interval is undefined.
date: 2056-04-07, spline end: 2056-04-05, curve id: 'eurswap_solver'
This often occurs when a curve is constructed with a final node date that aligns with the maturity of an instrument with a payment lag.
In the case that the instrument has a payment lag (e.g. a SOFR swap or ESTR swap or bond terminating on a non-business day) then a cashflow will occur after the maturity of the instrument.
The solution is to ensure that the final node date of the curve is changed to be beyond that expected payment date.
  warnings.warn(


In [14]:
# run pca from csv file with time series data for swap rates
swap_rate_pca = SwapRatePCAEstimator.from_csv( 
 swap_ts_csv_path = SWAP_TIME_SERIES_CSV_PATH 
)

In [10]:
# run mc simulation with scenarios for market/swap rates - 1000 scenarios, 1d shocks scaled to 30 day horizon
swap_rate_sim = SwapRateMCSimulator.from_csv(
  n_scenarios = 1000, 
  t_scale_factor = 30, 
  rng_seed = 1, 
  swap_ts_csv_path = SWAP_TIME_SERIES_CSV_PATH 
)

In [15]:
# create scenarios for curves calibrated to mc simulated market/swap rates 
scenarios = SwapCurveScenarioGenerator( 
    curve_date = curve_date, 
    swap_rate_sim = swap_rate_sim, 
    swap_rate_pca = swap_rate_pca
) 

In [ ]:
# calibrate the curves in each scenario of swap rates - required to generate PnL scenarios for mc CVaR calculation
# Note! For 1000 scenarios, this takes approx. 5-6 mins. 
scenarios.build_mc_scenario_curves( curve_id = 'eurswap' )

In [17]:
# create 30Y liability position 
lby_pos = LiabilityPosition(
  pos_id = 'liabilities 30Y', 
  pos_grp = 'liabilities', 
  currency = 'eur',   
  effective_dt = curve_date, 
  payment_schedule = liabilities_payment_schedule, 
  position_curve_builder = curve 
)

In [23]:
# create IRS positions 
irs2y = Euribor6mIRSPosition(
        pos_id = 'EURIBOR6M 2Y receiver', 
        pos_grp = 'irs hedge', 
        effective_dt = curve_date, 
        fixed_rate = 2.25, 
        notional = 1e6, 
        termination = '2Y', 
        is_receiver = True, 
        is_par_swap = True, 
        position_curve_builder = curve        
        )
  
irs5y = Euribor6mIRSPosition(
        pos_id = 'EURIBOR6M 5Y Receiver', 
        pos_grp = 'irs hedge', 
        effective_dt = curve_date, 
        fixed_rate = 2.25, 
        notional = 100e6, 
        termination = '5Y', 
        is_receiver = True, 
        position_curve_builder = curve, 
        is_par_swap = True)
  
irs10y = Euribor6mIRSPosition(
        pos_id = 'EURIBOR6M 10Y Receiver', 
        pos_grp = 'irs hedge', 
        effective_dt = curve_date, 
        fixed_rate = 2.25, 
        notional = 100e6, 
        termination = '10Y', 
        position_curve_builder = curve,         
        is_receiver = True, 
        is_par_swap = True)   

irs15y = Euribor6mIRSPosition(
        pos_id = 'EURIBOR6M 15Y Payer', 
        pos_grp = 'irs hedge', 
        effective_dt = curve_date, 
        fixed_rate = 2.25, 
        notional = -100e6, 
        termination = '15Y', 
        position_curve_builder = curve,         
        is_receiver = False, 
        is_par_swap = True)     
  
irs30y = Euribor6mIRSPosition(
        pos_id = 'EURIBOR6M 30Y Receiver', 
        pos_grp = 'irs hedge', 
        effective_dt = curve_date, 
        fixed_rate = 2.25, 
        notional = 100e6, 
        termination = '30Y', 
        position_curve_builder = curve,         
        is_receiver = True, 
        is_par_swap = True )  

In [22]:
# create liability portfolio 
lby_prtfl = LiabilityPortfolio(
    prtfl_id     = 'liabilities 30Y', 
    positions    = [lby_pos, irs2y, irs5y, irs10y, irs15y, irs30y],
    effective_dt = curve_date )

C:\Users\mduba\Desktop\Programming\Python\alm_ir_hedging_prototype\.venv\Lib\site-packages\rateslib\curves\curves.py:275: UserWarning: Evaluating points on a curve beyond the endpoint of the basic spline interval is undefined.
date: 2056-04-07, spline end: 2056-04-05, curve id: 'eurswap_solver'
This often occurs when a curve is constructed with a final node date that aligns with the maturity of an instrument with a payment lag.
In the case that the instrument has a payment lag (e.g. a SOFR swap or ESTR swap or bond terminating on a non-business day) then a cashflow will occur after the maturity of the instrument.
The solution is to ensure that the final node date of the curve is changed to be beyond that expected payment date.
  warnings.warn(
C:\Users\mduba\Desktop\Programming\Python\alm_ir_hedging_prototype\.venv\Lib\site-packages\rateslib\curves\curves.py:275: UserWarning: Evaluating points on a curve beyond the endpoint of the basic spline interval is undefined.
date: 2056-04-07, s

SUCCESS: `func_tol` reached after 0 iterations (levenberg_marquardt), `f_val`: 0.0, `time`: 0.0058s
SUCCESS: `func_tol` reached after 0 iterations (levenberg_marquardt), `f_val`: 0.0, `time`: 0.0074s
SUCCESS: `func_tol` reached after 0 iterations (levenberg_marquardt), `f_val`: 0.0, `time`: 0.0071s
SUCCESS: `func_tol` reached after 0 iterations (levenberg_marquardt), `f_val`: 0.0, `time`: 0.0075s


C:\Users\mduba\Desktop\Programming\Python\alm_ir_hedging_prototype\.venv\Lib\site-packages\rateslib\curves\curves.py:275: UserWarning: Evaluating points on a curve beyond the endpoint of the basic spline interval is undefined.
date: 2056-04-07, spline end: 2056-04-05, curve id: 'eurswap_solver'
This often occurs when a curve is constructed with a final node date that aligns with the maturity of an instrument with a payment lag.
In the case that the instrument has a payment lag (e.g. a SOFR swap or ESTR swap or bond terminating on a non-business day) then a cashflow will occur after the maturity of the instrument.
The solution is to ensure that the final node date of the curve is changed to be beyond that expected payment date.
  warnings.warn(
C:\Users\mduba\Desktop\Programming\Python\alm_ir_hedging_prototype\.venv\Lib\site-packages\rateslib\curves\curves.py:275: UserWarning: Evaluating points on a curve beyond the endpoint of the basic spline interval is undefined.
date: 2056-04-07, s

SUCCESS: `func_tol` reached after 0 iterations (levenberg_marquardt), `f_val`: 0.0, `time`: 0.0073s
SUCCESS: `func_tol` reached after 0 iterations (levenberg_marquardt), `f_val`: 0.0, `time`: 0.0073s


In [24]:
# simulate prtfl pnl - note that for 1000 PnL scenarios - calculation time is approx. 4-5 mins     
print("starting at: ", datetime.now())
lby_prtfl.simulate_pnl(mc_curve_scenarios = scenarios)
print("ended at: ", datetime.now())

starting at:  2026-05-05 20:26:48.039705
ended at:  2026-05-05 20:29:39.974105


In [25]:
# calculate CVaR and related metrics
lby_prtfl.calc_cvar(0.95)

In [26]:
lby_prtfl.sim_pos_pnl
lby_prtfl.sim_pos_grp_pnl
lby_prtfl.sim_prtfl_pnl 
lby_prtfl.prtfl_cvar
lby_prtfl.pos_grp_cvar
lby_prtfl.pos_cvar
lby_prtfl.cvar_scenarios
lby_prtfl.pv01_delta_cvar_attr

,Portfolio ID,Measure,Total,2Y,3Y,5Y,7Y,10Y,12Y,15Y,20Y,25Y,30Y
0,liabilities 30Y,PV01 Delta Contribution in Tail Scenarios,-6.265160e+08,221578.31911,-305843.09548,1316632.583593,-7426804.010407,-15249364.056163,-14123969.527647,-93955352.042347,-166368746.198439,-238349748.908965,-92274393.522742
1,liabilities 30Y,Same but as % of MC CVaR,9.461372e+01,-0.033462,0.046187,-0.198832,1.121564,2.302893,2.13294,14.188728,25.124284,35.994542,13.934877
